# PhenoAssistant Case 2 on Microsoft Agent Framework

CPU-side migration of `case2.ipynb`.

The original notebook has two stages:

1. compute potato projected leaf area (PLA) from 32 plant images;
2. compare manual and algorithm-derived leaf area as predictors of dried weight.

Task 1 requires the vision/segmentation phenotype-extraction pathway and is
therefore an explicit GPU-deferred boundary. This notebook does **not** claim
that MAF generated the existing phenotype table. Instead, it resumes from the
tracked legacy artifact `results/Case2/potato_phenotypes.csv`.

Task 2 is migrated through the production MAF
`compare_linear_relationships` capability.

In [ ]:
from __future__ import annotations

import inspect
import os
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import linregress

from phenoassistant_maf import (
    OpenRouterSettings,
    build_application,
    create_chat_client,
    run_application,
)

ROOT = Path.cwd()

METADATA = ROOT / "data/potato_metadata.csv"
CASE2_DATA = ROOT / "results/Case2/potato_phenotypes.csv"
OUTPUT_DIR = ROOT / "results/maf_case2"

MANUAL_PLOT = OUTPUT_DIR / "potato_manual.png"
ALGORITHM_PLOT = OUTPUT_DIR / "potato_algorithm.png"

assert METADATA.is_file()
assert CASE2_DATA.is_file()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata = pd.read_csv(METADATA)
phenotypes = pd.read_csv(CASE2_DATA)

assert len(metadata) == 32
assert len(phenotypes) == 32

required_columns = {
    "file_name",
    "projected_leaf_area",
    "manual_leaf_area",
    "manual_dried_weight",
}

assert required_columns.issubset(phenotypes.columns)

print("CASE2_METADATA_ROWS=", len(metadata))
print("CASE2_PHENOTYPE_ROWS=", len(phenotypes))
print("CASE2_DATA=", CASE2_DATA.relative_to(ROOT))

## Original Task 1 — phenotype extraction deferred at the GPU boundary

The original task starts with `data/potato_metadata.csv`, runs vision inference
on 32 potato images, computes projected leaf area, and writes the resulting
phenotypes.

That vision execution is not simulated in the CPU migration.

All 32 original image references are present in the repository, and the
repository already contains the tracked legacy result
`results/Case2/potato_phenotypes.csv`. The analytical migration below resumes
from that artifact.

This distinction is intentional:

- the existing phenotype CSV is preserved as provenance;
- MAF does not claim to have regenerated it;
- raw segmentation and phenotype extraction remain deferred until the
  canonical Nottingham GPU environment is available.

In [ ]:
if not os.environ.get("OPENROUTER_API_KEY"):
    raise RuntimeError("Set OPENROUTER_API_KEY before running this cell.")

os.environ.setdefault("OPENROUTER_MODEL", "openrouter/free")

settings = OpenRouterSettings.from_env()
client = create_chat_client(settings)


def forbidden_calculator(
    a: int,
    b: int,
    operator: str,
) -> int:
    raise AssertionError(
        "Case 2 selected calculator unexpectedly."
    )


def forbidden_anova(
    data_path: str,
    descriptor: str,
    within_subject_factor: str,
    between_subject_factor: str,
    subject_id: str,
    save_path: str | None = None,
) -> list[dict[str, Any]]:
    raise AssertionError(
        "Case 2 selected ANOVA unexpectedly."
    )


def forbidden_tukey(
    data_path: str,
    descriptor: str,
    between_subject_factor: str,
    subject_id: str,
    save_path: str | None = None,
) -> list[dict[str, Any]]:
    raise AssertionError(
        "Case 2 selected Tukey unexpectedly."
    )


def forbidden_aggregate(
    data_path: str,
    operation: str,
    value_column: str,
    filter_column: str | None,
    filter_value: str | None,
) -> dict[str, Any]:
    raise AssertionError(
        "Case 2 selected CSV aggregation unexpectedly."
    )


def case2_regression(
    data_path: str,
    x_column: str,
    y_column: str,
    plot_path: str,
) -> dict[str, float]:
    # Deterministic CPU regression implementation behind the MAF adapter.
    data = pd.read_csv(data_path)

    x = pd.to_numeric(
        data[x_column],
        errors="raise",
    )
    y = pd.to_numeric(
        data[y_column],
        errors="raise",
    )

    result = linregress(x, y)

    destination = Path(plot_path)
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    figure, axis = plt.subplots()

    axis.scatter(
        x,
        y,
        label="observations",
    )

    fitted = (
        result.intercept
        + result.slope * x
    )

    order = x.argsort()

    axis.plot(
        x.iloc[order],
        fitted.iloc[order],
        label="linear fit",
    )

    axis.set_xlabel(x_column)
    axis.set_ylabel(y_column)
    axis.set_title("Linear fit")
    axis.legend()

    figure.tight_layout()
    figure.savefig(destination)
    plt.close(figure)

    return {
        "slope": float(result.slope),
        "intercept": float(result.intercept),
        "r_value": float(result.rvalue),
    }


application = build_application(
    client=client,
    data_path=str(CASE2_DATA),
    calculator_callable=forbidden_calculator,
    anova_callable=forbidden_anova,
    tukey_callable=forbidden_tukey,
    regression_callable=case2_regression,
    aggregate_callable=forbidden_aggregate,
    first_plot_path=str(MANUAL_PLOT),
    second_plot_path=str(ALGORITHM_PLOT),
)

print("MODEL=", settings.model)
print("TOOLS=", application.registry.names)

In [ ]:
def response_text(response: Any) -> str:
    messages = getattr(
        response,
        "messages",
        None,
    )

    if messages:
        text = getattr(
            messages[-1],
            "text",
            None,
        )

        if isinstance(text, str):
            return text.strip()

    return str(response).strip()

In [ ]:
# Original Task 2:
# compare manual and algorithm-derived leaf area as predictors of dried weight.

comparison_response = await run_application(
    application,
    '''
    Using the trusted Case 2 potato phenotype dataset, call
    compare_linear_relationships exactly once.

    Compare:
    1. manual_leaf_area against manual_dried_weight; and
    2. projected_leaf_area against manual_dried_weight.

    Report the fitted linear equations and Pearson r values for both
    relationships, identify which leaf-area measurement is more strongly
    associated with dried weight, and state what that evidence implies about
    the reliability of model-derived projected leaf area relative to the
    manually measured leaf area.

    Summarise only evidence returned by the tool.
    ''',
)

print(response_text(comparison_response))

In [ ]:
for path in (
    MANUAL_PLOT,
    ALGORITHM_PLOT,
):
    assert path.is_file()
    assert path.stat().st_size > 0

print("CASE2_MAF_CPU_WORKFLOW_COMPLETE")


async def close_provider(value: Any) -> None:
    for candidate in (
        value,
        getattr(value, "client", None),
        getattr(value, "_client", None),
        getattr(value, "_openai_client", None),
    ):
        if candidate is None:
            continue

        for name in (
            "aclose",
            "close",
        ):
            method = getattr(
                candidate,
                name,
                None,
            )

            if callable(method):
                result = method()

                if inspect.isawaitable(result):
                    await result

                return


await close_provider(client)

print("OPENROUTER_CLIENT_CLOSED")